In [ ]:
import os, shutil, subprocess
repo_dir = "slot-imputation"
if os.path.isdir(repo_dir):
    shutil.rmtree(repo_dir)
subprocess.run(["git", "clone", "https://github.com/jdbohrman/slot-imputation.git"], check=True)


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("slot-imputation/src"))
# verify
from slot_impute import transformer; print("slot_impute imported OK")


In [ ]:
import json, math, time
import torch
import torch.nn.functional as F

from slot_impute.transformer import TransformerConfig, MiniGPT2
from slot_impute.data import load_wikitext_batches, wikitext_batch_iterator
from slot_impute.train_transformer import train_model
from slot_impute.impute_mlp import impute_mlp_to_target
from slot_impute.validate_mlp import compute_ppl, weight_distance, mlp_ablation_gap, convergence_head_start, compute_cost_savings, compute_dollar_savings

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Phase 1 — Quick Test

Tiny model proves the upsampling works before committing to a long run.

In [ ]:
cfg = TransformerConfig(d_model=128, d_ff=128, n_heads=4, n_layers=2, max_seq_len=128, vocab_size=50257)
model = MiniGPT2(cfg).to(DEVICE)
print(f"Model: d_model={cfg.d_model} d_ff={cfg.d_ff} params={model.num_parameters():,}")

batches = load_wikitext_batches(batch_size=16, seq_len=128, split="train", subset_tokens=32768, device=DEVICE)
print(f"Data: {len(batches)} batches")

t0 = time.time()
losses = train_model(model, batches, DEVICE, steps=300, lr=6e-4, seed=42, warmup_steps=30, grad_clip=1.0)
print(f"Training done ({time.time() - t0:.0f}s). Final loss: {sum(losses[-30:])/30:.4f}")

In [ ]:
val = load_wikitext_batches(batch_size=16, seq_len=128, split="validation", device=DEVICE)

imputed = impute_mlp_to_target(model, 256, val, num_act_batches=5)
print(f"Imputed d_ff={imputed.config.d_ff}, params={imputed.num_parameters():,}")

In [ ]:
@torch.no_grad()
def ppl(m, batches, n=5):
    m.eval(); m.to(DEVICE)
    total, cnt = 0.0, 0
    for b in batches[:n]:
        x = b["input_ids"].to(DEVICE); y = b["labels"].to(DEVICE)
        loss = F.cross_entropy(m(x).contiguous().view(-1, m.config.vocab_size), y.contiguous().view(-1))
        total += loss.item(); cnt += 1
    return math.exp(min(total / max(cnt, 1), 20))

ppl_src  = ppl(model,   val)
ppl_imp  = ppl(imputed, val)
ppl_rand = ppl(MiniGPT2(imputed.config), val)

print(f"d_ff=128 source:  {ppl_src:.1f}")
print(f"d_ff=256 imputed: {ppl_imp:.1f}")
print(f"d_ff=256 random:  {ppl_rand:.1f}")
print()
if ppl_imp < ppl_src:  # imputed beats source model despite no training at d_ff=256
    print(f"PASSED — imputed PPL is {ppl_src - ppl_imp:.0f} below source ({ppl_imp/ppl_src:.2f}x)")
else:
    print(f"WEAK — imputed PPL {ppl_imp - ppl_src:.0f} above source (imputation didn't transfer)")

In [ ]:
# Projected cost savings at GPT-2 scale (d_model=768, n_layers=12)
ds_proj = compute_dollar_savings(2048, 4096, 768, 12,
    steps=20000, gpu_cost_per_hour=2.0)
print(f"MLP FLOPs ratio:  {ds_proj['mlp_flops_ratio']}x")
print(f"Block FLOPs ratio: {ds_proj['block_flops_ratio']}x  (attention unchanged, MLP 2x)")
print(f"Compute savings:   {ds_proj['compute_savings_pct']}%")
print(f"")
print(f"To train this 12-layer model on {ds_proj['gpu']} for {ds_proj['steps']:,} steps:")
print(f"  Slim ({ds_proj['d_ff_small']}): {ds_proj['training_time_slim']}")
print(f"  Full ({ds_proj['d_ff_large']}): {ds_proj['training_time_full']}")


## Phase 2 — Full Experiment

GPT-2 scale: d_model=768, n_layers=12, d_ff=2048→4096. ~6-8h on T4 GPU.

In [ ]:
cfg_src = TransformerConfig(d_model=768, d_ff=2048, n_heads=12, n_layers=12, max_seq_len=1024)
cfg_gt  = TransformerConfig(d_model=768, d_ff=4096, n_heads=12, n_layers=12, max_seq_len=1024)
model_src = MiniGPT2(cfg_src).to(DEVICE)
model_gt  = MiniGPT2(cfg_gt).to(DEVICE)
print(f"Source (d_ff=2048):      {model_src.num_parameters():,} params")
print(f"Ground truth (d_ff=4096): {model_gt.num_parameters():,} params")

In [ ]:
batches = load_wikitext_batches(batch_size=8, seq_len=1024, split="train", device=DEVICE)
print(f"Train data: {len(batches)} batches")

print("Training source (d_ff=2048)...")
t_src_start = time.time()
train_model(model_src, batches, DEVICE, steps=20000, lr=6e-4, seed=42, warmup_steps=1000, grad_clip=1.0, save_dir="checkpoints_dff", checkpoint_tag="source")
t_src_sec = time.time() - t_src_start
ms_per_step_src = t_src_sec * 1000 / 20000
print(f"  Source training time: {t_src_sec:.0f}s ({ms_per_step_src:.1f} ms/step)")

print("Training ground truth (d_ff=4096)...")
t_gt_start = time.time()
train_model(model_gt, batches, DEVICE, steps=20000, lr=6e-4, seed=42, warmup_steps=1000, grad_clip=1.0, save_dir="checkpoints_dff", checkpoint_tag="ground_truth")
t_gt_sec = time.time() - t_gt_start
ms_per_step_gt = t_gt_sec * 1000 / 20000
print(f"  Ground truth training time: {t_gt_sec:.0f}s ({ms_per_step_gt:.1f} ms/step)")


In [ ]:
val = load_wikitext_batches(batch_size=8, seq_len=1024, split="validation", device=DEVICE)
imputed = impute_mlp_to_target(model_src, 4096, val, num_act_batches=10)

ppl_src  = compute_ppl(model_src, val, DEVICE, 20)
ppl_imp  = compute_ppl(imputed,   val, DEVICE, 20)
ppl_gt   = compute_ppl(model_gt,  val, DEVICE, 20)
ppl_rand = ppl(MiniGPT2(imputed.config), val, 20)

wd = weight_distance(imputed, model_gt)
abl = mlp_ablation_gap(model_src, val, DEVICE, 10)
conv = convergence_head_start(imputed, val, DEVICE, 200)
ds = compute_dollar_savings(2048, 4096, 768, 12, steps=20000,
    ms_per_step_small=ms_per_step_src, ms_per_step_large=ms_per_step_gt, gpu_cost_per_hour=2.0)

print(f"PPL — src:{ppl_src:.1f} imp:{ppl_imp:.1f} gt:{ppl_gt:.1f} rand:{ppl_rand:.1f}")
print(f"Weight — W1 mse:{wd['W1_mse']:.4f} cos:{wd['W1_cosine']:.3f} W2 mse:{wd['W2_mse']:.4f} cos:{wd['W2_cosine']:.3f}")
print(f"MLP gap — with:{abl['ppl_with_mlp']:.1f} without:{abl['ppl_without_mlp']:.1f} gap:{abl['mlp_gap']:.1f}")
print(f"Convergence — speedup:{conv['speedup_ratio']:.3f} ({conv['speedup_steps']}/200)")
print(f"Cost \u2014 {ds['compute_savings_pct']}% compute savings ({ds['block_flops_ratio']}x FLOPs ratio)")

print()
print(f"Dollar analysis ({ds['gpu']}):")
print(f"  Train slim ({ds['d_ff_small']}): {ds['training_time_slim']} \u2192 {ds['cost_train_slim']}")
print(f"  Train full ({ds['d_ff_large']}): {ds['training_time_full']} \u2192 {ds['cost_train_full']}")
print(f"  Imputation: {ds['cost_imputation']}")
print(f"  Savings: {ds['cost_savings']} ({ds['compute_savings_pct']}%)")
json.dump({
    "ppl": {"source": ppl_src, "imputed": ppl_imp, "ground_truth": ppl_gt, "random": ppl_rand},
    "weight_distance": wd, "ablation": abl, "convergence": conv, "cost": ds,
}, open("dff_results.json", "w"), indent=2, default=str)
print("Saved dff_results.json")

## Phase 3 — Cost Scaling Analysis

Compute savings are **architecture-dependent but scale-invariant** —
they depend only on `d_ff/d_model` and the upsample ratio, not on total params,
batch size, GPU type, or training steps.

**Key formula**: for `d_ff = ff × d_model` and upsample factor `u`:
```
block_ratio = (2 + ff×u) / (2 + ff)
savings %   = (1 − 1/block_ratio) × 100
```
Absolute dollar savings = training budget × savings % / 100.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook", font_scale=1.1)

# Analytical: block_ratio = (2*dm + dff_large) / (2*dm + dff_small)
# With d_ff = ff * dm:   ratio = (2 + ff*u) / (2 + ff)
#                        savings = (1 - 1/ratio) * 100
# dm cancels out — savings % depends only on ff and u, not model size.

ff_vals   = np.linspace(1, 16, 200)  # d_ff / d_model
upsamples = [2, 3, 4, 6, 8]
budgets   = np.array([1e3, 3e3, 1e4, 3e4, 1e5, 3e5, 1e6])  # $1K → $1M
ref_archs = [
    ("d_ff=2×d_model (small MLP)", 2),
    ("d_ff=4×d_model (GPT-2/3)",    4),
    ("d_ff=8×d_model (LLaMA)",      8),
]

palette = sns.color_palette("magma_r", len(upsamples))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# --- Plot 1: Savings % vs d_ff/d_model ---
for u, color in zip(upsamples, palette):
    ratio = (2 + ff_vals * u) / (2 + ff_vals)
    savings = (1 - 1 / ratio) * 100
    ax1.plot(ff_vals, savings, color=color, lw=2.5, label=f"{u}× upsampling")

# Mark reference architectures
for label, ff, va in [("GPT-2/3", 4, "top"), ("LLaMA", 8, "bottom"), ("d_ff=2×", 2, "bottom")]:
    ax1.axvline(ff, color="gray", ls="--", alpha=0.3, lw=1)
    for idx, u in enumerate(upsamples):
        r = (2 + ff * u) / (2 + ff)
        s = (1 - 1 / r) * 100
        ax1.scatter(ff, s, color=palette[idx], s=50, zorder=5, edgecolors="white", linewidths=0.5)
    # Label at top of first line
    r_max = (2 + ff * upsamples[-1]) / (2 + ff)
    s_max = (1 - 1 / r_max) * 100
    y = s_max + (4 if va == "top" else -6)
    ax1.text(ff, y, label, ha="center", fontsize=8.5, fontweight="bold",
             bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="gray", alpha=0.7))

ax1.set_xlabel("d_ff / d_model ratio", fontsize=12)
ax1.set_ylabel("Compute Savings (%)", fontsize=12)
ax1.set_title("Savings % — Architecture Only", fontweight="bold", fontsize=13)
ax1.legend(fontsize=8, loc="lower right")
ax1.set_xlim(1, 16)
ax1.set_ylim(0, 92)

# --- Plot 2: Absolute savings × training budget ---
styles = ["-", "--", "-."]
for (label, ff), ls in zip(ref_archs, styles):
    for u in [2, 4, 8]:
        ratio = (2 + ff * u) / (2 + ff)
        pct = (1 - 1 / ratio) * 100
        ax2.plot(budgets, budgets * pct / 100, marker="o", markersize=4,
                 lw=1.8, linestyle=ls,
                 label=f"{label.split('(')[0].strip()}, {u}×")

ax2.set_xscale("log")
ax2.set_yscale("log")
ax2.set_xlabel("Training Budget ($)", fontsize=12)
ax2.set_ylabel("Absolute Savings ($)", fontsize=12)
ax2.set_title("Dollar Savings × Training Budget", fontweight="bold", fontsize=13)
ax2.legend(fontsize=7, ncol=2, loc="upper left")

plt.tight_layout()
plt.show()

# --- Summary table ---
print("\nSavings % (architecture only, no batch/GPU dependency):")
print(f"{'d_ff/dm':<10}", end="")
for u in upsamples:
    print(f"{u}× upsample".rjust(16), end="")
print()
print("-" * (10 + 16*len(upsamples)))
for ff in [1, 2, 4, 8, 16]:
    print(f"{ff:<10}", end="")
    for u in upsamples:
        r = (2 + ff * u) / (2 + ff)
        s = (1 - 1 / r) * 100
        print(f"{s:>14.1f}%", end="  ")
    print()

print("\n\nAbsolute savings @ reference budgets:")
budget_labels = ["$1K", "$3K", "$10K", "$30K", "$100K", "$300K", "$1M"]
print(f"{'Budget':<10}", end="")
for label, ff in ref_archs:
    short = label.split("(")[0].strip()
    for u in [2, 4, 8]:
        col = f"{short} {u}×"
        print(f"{col:>14s}".replace("d_",""), end="")
print()
print("-" * (10 + 14 * 9))
for b, bl in zip(budgets, budget_labels):
    print(f"{bl:<10}", end="")
    for label, ff in ref_archs:
        for u in [2, 4, 8]:
            r = (2 + ff * u) / (2 + ff)
            s = (1 - 1 / r) * 100
            amt = b * s / 100
            if amt >= 1000:
                print(f"  ${amt/1000:>8.1f}K", end="  ")
            else:
                print(f"  ${amt:>8,.0f}", end="  ")
    print()
